# Advanced Problems with Solutions: Named Tuples

This notebook contains advanced practice problems on Python `namedtuple` objects from the `collections` module.

Topics covered:

- creating named tuple classes
- field names and validation
- positional access vs attribute access
- unpacking and iteration
- immutability and `_replace()`
- `_fields`, `_asdict()`, and `_make()`
- named tuples as lightweight data records
- converting plain tuples into named tuples
- designing readable data-processing APIs

## Best-Practice Notes

1. Use a named tuple when records are small, immutable, and field names improve readability.
2. Prefer attribute access, such as `point.x`, over unclear indexing, such as `point[0]`.
3. Use `_replace()` to create modified copies instead of trying to mutate a named tuple.
4. Use `_asdict()` when converting records to dictionaries or JSON-friendly structures.
5. Use `_fields` when writing generic validation or conversion utilities.
6. Avoid invalid field names unless you intentionally use `rename=True`.

## Setup

In [1]:
from collections import namedtuple
from math import sqrt
from statistics import mean
import json

## Problem 1 — Replace a Boilerplate Class with a Named Tuple

A beginner wrote this class:

```python
class Point3D:
    def __init__(self, x, y, z):
        self.x = x
        self.y = y
        self.z = z
```

Replace it with a `namedtuple` called `Point3D`.

Then create two equal points and verify that:

- their representation is readable
- equality works by value
- the object is also an instance of `tuple`
- the coordinates can be accessed by name and by index

In [2]:
Point3D = namedtuple("Point3D", "x y z")

p1 = Point3D(10, 20, 30)
p2 = Point3D(10, 20, 30)

assert repr(p1) == "Point3D(x=10, y=20, z=30)"
assert p1 == p2
assert isinstance(p1, tuple)
assert p1.x == 10
assert p1[0] == 10
assert p1.z == 30
assert tuple(p1) == (10, 20, 30)

p1

Point3D(x=10, y=20, z=30)

## Problem 2 — Vector Operations on Named Tuples

Create a generic function `dot_product(a, b)` that works with any two equal-length tuple-like objects, including named tuples.

Requirements:

- use `zip`
- raise `ValueError` if the vectors have different lengths
- work with both plain tuples and named tuples
- return the mathematical dot product

In [3]:
def dot_product(a, b):
    if len(a) != len(b):
        raise ValueError("vectors must have the same length")

    return sum(x * y for x, y in zip(a, b))


Point2D = namedtuple("Point2D", "x y")
Point4D = namedtuple("Point4D", "i j k l")

assert dot_product(Point2D(1, 2), Point2D(10, 20)) == 50
assert dot_product((1, 2, 3), (10, 20, 30)) == 140
assert dot_product(Point4D(1, 1, 1, 10), Point4D(2, 2, 2, 10)) == 106

try:
    dot_product((1, 2), (1, 2, 3))
except ValueError as ex:
    assert "same length" in str(ex)
else:
    raise AssertionError("Expected ValueError for mismatched vector lengths")

dot_product(Point2D(1, 2), Point2D(10, 20))

50

## Problem 3 — Immutable Updates with `_replace()`

A stock record is represented as:

```python
Stock(symbol, year, month, day, open, high, low, close)
```

Write `adjust_close(stock, new_close)` that returns a new stock record with the updated close value.

Requirements:

- do not mutate the original record
- use `_replace()`
- reject negative close values

In [4]:
Stock = namedtuple("Stock", "symbol year month day open high low close")


def adjust_close(stock, new_close):
    if new_close < 0:
        raise ValueError("close value cannot be negative")

    return stock._replace(close=new_close)


djia = Stock("DJIA", 2018, 1, 25, 26313, 26458, 26260, 26393)
updated = adjust_close(djia, 27000)

assert djia.close == 26393
assert updated.close == 27000
assert updated.symbol == "DJIA"
assert updated is not djia

try:
    adjust_close(djia, -1)
except ValueError as ex:
    assert "negative" in str(ex)
else:
    raise AssertionError("Expected ValueError for negative close")

updated

Stock(symbol='DJIA', year=2018, month=1, day=25, open=26313, high=26458, low=26260, close=27000)

## Problem 4 — Convert Raw Tuples into Named Tuples

You receive raw city records in this format:

```python
(name, country, population)
```

Create a `City` named tuple and write `convert_cities(raw_records)`.

Requirements:

- return a tuple of `City` objects
- validate that each record has exactly 3 values
- validate that population is a non-negative integer
- strip whitespace from city and country names

In [5]:
City = namedtuple("City", "name country population")


def convert_cities(raw_records):
    cities = []

    for index, record in enumerate(raw_records):
        try:
            name, country, population = record
        except ValueError as exc:
            raise ValueError(f"record #{index} must have exactly 3 fields") from exc

        if not isinstance(name, str) or not name.strip():
            raise ValueError(f"record #{index} has invalid city name")

        if not isinstance(country, str) or not country.strip():
            raise ValueError(f"record #{index} has invalid country")

        if not isinstance(population, int) or population < 0:
            raise ValueError(f"record #{index} has invalid population")

        cities.append(City(name.strip(), country.strip(), population))

    return tuple(cities)


raw = [
    (" London ", " UK ", 8_780_000),
    ("New York", "USA", 8_500_000),
    ("Beijing", "China", 21_000_000)
]

cities = convert_cities(raw)

assert cities == (
    City("London", "UK", 8_780_000),
    City("New York", "USA", 8_500_000),
    City("Beijing", "China", 21_000_000)
)
assert cities[0].name == "London"

cities

(City(name='London', country='UK', population=8780000),
 City(name='New York', country='USA', population=8500000),
 City(name='Beijing', country='China', population=21000000))

## Problem 5 — Serialize Named Tuples with `_asdict()`

Write `cities_to_json(cities)` that converts a sequence of `City` records into a JSON string.

Requirements:

- use `_asdict()`
- preserve field names
- sort cities by population descending before serializing
- return a JSON-formatted string

In [6]:
def cities_to_json(cities):
    sorted_cities = sorted(cities, key=lambda city: city.population, reverse=True)
    dictionaries = [dict(city._asdict()) for city in sorted_cities]
    return json.dumps(dictionaries, indent=2)


json_result = cities_to_json(cities)
decoded = json.loads(json_result)

assert decoded[0]["name"] == "Beijing"
assert decoded[0]["population"] == 21_000_000
assert set(decoded[0].keys()) == {"name", "country", "population"}

print(json_result)

[
  {
    "name": "Beijing",
    "country": "China",
    "population": 21000000
  },
  {
    "name": "London",
    "country": "UK",
    "population": 8780000
  },
  {
    "name": "New York",
    "country": "USA",
    "population": 8500000
  }
]


## Problem 6 — Create Named Tuples Dynamically with Field Validation

Write `safe_namedtuple(type_name, field_names)` that creates a named tuple class.

Requirements:

- reject empty type names
- reject empty field-name collections
- reject duplicate field names
- reject field names that start with `_`
- return the generated named tuple class

Do not use `rename=True` in this problem. The point is to validate explicitly.

In [7]:
def safe_namedtuple(type_name, field_names):
    if not isinstance(type_name, str) or not type_name.strip():
        raise ValueError("type_name must be a non-empty string")

    field_names = tuple(field_names)

    if not field_names:
        raise ValueError("field_names cannot be empty")

    if len(set(field_names)) != len(field_names):
        raise ValueError("field names cannot contain duplicates")

    for field in field_names:
        if not isinstance(field, str) or not field:
            raise ValueError("all field names must be non-empty strings")
        if field.startswith("_"):
            raise ValueError("field names cannot start with an underscore")

    return namedtuple(type_name.strip(), field_names)


Book = safe_namedtuple("Book", ["title", "author", "year"])
book = Book("Fluent Python", "Luciano Ramalho", 2015)

assert book.title == "Fluent Python"
assert Book._fields == ("title", "author", "year")

for invalid_fields in [["x", "x"], ["x", "_y"], []]:
    try:
        safe_namedtuple("Bad", invalid_fields)
    except ValueError:
        pass
    else:
        raise AssertionError("Expected ValueError")

book

Book(title='Fluent Python', author='Luciano Ramalho', year=2015)

## Problem 7 — Use `rename=True` for Dirty Input Schemas

External systems sometimes provide bad field names.

Create a named tuple class called `Person` using these fields:

```python
["firstname", "lastname", "_age", "class"]
```

Requirements:

- use `rename=True`
- create a person record
- inspect `_fields`
- demonstrate how invalid fields were automatically renamed

In [8]:
Person = namedtuple(
    "Person",
    ["firstname", "lastname", "_age", "class"],
    rename=True
)

person = Person("Ada", "Lovelace", 36, "mathematician")

assert Person._fields == ("firstname", "lastname", "_2", "_3")
assert person.firstname == "Ada"
assert person.lastname == "Lovelace"
assert person._2 == 36
assert person._3 == "mathematician"

Person._fields, person

(('firstname', 'lastname', '_2', '_3'),
 Person(firstname='Ada', lastname='Lovelace', _2=36, _3='mathematician'))

## Problem 8 — Generic Named Tuple Validator Using `_fields`

Write `validate_required_fields(record, required_fields)`.

Requirements:

- accept any named tuple instance
- verify that all `required_fields` exist in `record._fields`
- verify that the corresponding values are not `None`
- return `True` if valid
- raise helpful exceptions otherwise

In [9]:
def validate_required_fields(record, required_fields):
    if not hasattr(record, "_fields"):
        raise TypeError("record must be a named tuple instance")

    missing_fields = set(required_fields) - set(record._fields)

    if missing_fields:
        raise ValueError(f"missing fields from record type: {sorted(missing_fields)}")

    none_fields = [field for field in required_fields if getattr(record, field) is None]

    if none_fields:
        raise ValueError(f"required fields cannot be None: {none_fields}")

    return True


User = namedtuple("User", "id username email")
valid_user = User(1, "ada", "ada@example.com")
invalid_user = User(2, "grace", None)

assert validate_required_fields(valid_user, ["id", "username", "email"]) is True

try:
    validate_required_fields(invalid_user, ["id", "email"])
except ValueError as ex:
    assert "None" in str(ex)
else:
    raise AssertionError("Expected ValueError for None email")

try:
    validate_required_fields((1, 2, 3), ["id"])
except TypeError as ex:
    assert "named tuple" in str(ex)
else:
    raise AssertionError("Expected TypeError for plain tuple")

validate_required_fields(valid_user, ["id", "username", "email"])

True

## Problem 9 — Build Records from Iterables with `_make()`

Named tuple classes provide `_make(iterable)` for building instances from iterables.

Create an `Employee` named tuple:

```python
Employee(id, name, department, salary)
```

Write `load_employees(rows)` that:

- uses `Employee._make(row)`
- returns a tuple of employees
- raises a helpful `ValueError` if a row has the wrong number of fields

In [10]:
Employee = namedtuple("Employee", "id name department salary")


def load_employees(rows):
    employees = []

    for index, row in enumerate(rows):
        try:
            employees.append(Employee._make(row))
        except TypeError as exc:
            raise ValueError(
                f"row #{index} must contain exactly {len(Employee._fields)} fields"
            ) from exc

    return tuple(employees)


rows = [
    (1, "Ada", "Engineering", 120000),
    (2, "Grace", "Research", 125000),
]

employees = load_employees(rows)

assert employees == (
    Employee(1, "Ada", "Engineering", 120000),
    Employee(2, "Grace", "Research", 125000)
)

try:
    load_employees([(3, "Linus")])
except ValueError as ex:
    assert "exactly 4 fields" in str(ex)
else:
    raise AssertionError("Expected ValueError for short row")

employees

(Employee(id=1, name='Ada', department='Engineering', salary=120000),
 Employee(id=2, name='Grace', department='Research', salary=125000))

## Problem 10 — Payroll Analytics with Named Tuples

Using the `Employee` records from the previous problem, write `payroll_report(employees)`.

Return a named tuple called `PayrollReport` with fields:

```python
employee_count, total_salary, average_salary, highest_paid
```

Requirements:

- `highest_paid` should be the full `Employee` record
- raise `ValueError` for an empty employee collection
- use named fields in the implementation

In [11]:
PayrollReport = namedtuple(
    "PayrollReport",
    "employee_count total_salary average_salary highest_paid"
)


def payroll_report(employees):
    employees = tuple(employees)

    if not employees:
        raise ValueError("employees cannot be empty")

    total_salary = sum(employee.salary for employee in employees)
    average_salary = total_salary / len(employees)
    highest_paid = max(employees, key=lambda employee: employee.salary)

    return PayrollReport(
        employee_count=len(employees),
        total_salary=total_salary,
        average_salary=average_salary,
        highest_paid=highest_paid
    )


report = payroll_report(employees)

assert report.employee_count == 2
assert report.total_salary == 245000
assert report.average_salary == 122500
assert report.highest_paid == Employee(2, "Grace", "Research", 125000)

try:
    payroll_report([])
except ValueError as ex:
    assert "empty" in str(ex)
else:
    raise AssertionError("Expected ValueError for empty employees")

report

PayrollReport(employee_count=2, total_salary=245000, average_salary=122500.0, highest_paid=Employee(id=2, name='Grace', department='Research', salary=125000))

## Problem 11 — Geometric Records with Named Tuples

Create these named tuples:

```python
Point(x, y)
Circle(center, radius)
```

Where `center` is a `Point`.

Write:

```python
circle_area(circle)
contains_point(circle, point)
move_circle(circle, dx, dy)
```

Requirements:

- use named attributes
- `move_circle` must return a new circle
- do not mutate the original circle
- reject negative radius when constructing test data

In [12]:
Point = namedtuple("Point", "x y")
Circle = namedtuple("Circle", "center radius")


def make_circle(center, radius):
    if radius < 0:
        raise ValueError("radius cannot be negative")
    return Circle(center=center, radius=radius)


def circle_area(circle):
    return 3.141592653589793 * circle.radius ** 2


def contains_point(circle, point):
    dx = point.x - circle.center.x
    dy = point.y - circle.center.y
    distance = sqrt(dx * dx + dy * dy)
    return distance <= circle.radius


def move_circle(circle, dx, dy):
    new_center = Point(circle.center.x + dx, circle.center.y + dy)
    return circle._replace(center=new_center)


circle = make_circle(Point(0, 0), 10)
moved = move_circle(circle, 5, -2)

assert contains_point(circle, Point(3, 4)) is True
assert contains_point(circle, Point(20, 20)) is False
assert moved.center == Point(5, -2)
assert circle.center == Point(0, 0)

try:
    make_circle(Point(0, 0), -1)
except ValueError as ex:
    assert "radius" in str(ex)
else:
    raise AssertionError("Expected ValueError for negative radius")

moved

Circle(center=Point(x=5, y=-2), radius=10)

## Problem 12 — Schema Evolution with Defaults

A user record originally had three fields:

```python
User(id, username, email)
```

A new version adds an `is_active` field.

Create:

```python
UserV2(id, username, email, is_active=True)
```

Then write `upgrade_user(old_user)` that converts an old user record into a `UserV2` record.

Requirements:

- use `defaults` when creating `UserV2`
- preserve old values
- set `is_active=True` by default
- accept only records that contain `id`, `username`, and `email`

In [13]:
UserV1 = namedtuple("UserV1", "id username email")
UserV2 = namedtuple("UserV2", "id username email is_active", defaults=[True])


def upgrade_user(old_user):
    required = {"id", "username", "email"}

    if not hasattr(old_user, "_fields") or not required.issubset(old_user._fields):
        raise TypeError("old_user must have id, username, and email fields")

    return UserV2(
        id=old_user.id,
        username=old_user.username,
        email=old_user.email
    )


old_user = UserV1(1, "ada", "ada@example.com")
new_user = upgrade_user(old_user)

assert new_user == UserV2(1, "ada", "ada@example.com", True)
assert new_user.is_active is True

default_user = UserV2(2, "grace", "grace@example.com")
assert default_user.is_active is True

new_user

UserV2(id=1, username='ada', email='ada@example.com', is_active=True)

## Problem 13 — Named Tuple Records from Dictionaries

You receive API data as dictionaries:

```python
{"symbol": "AAPL", "price": 190.5, "currency": "USD"}
```

Create a `Quote` named tuple with fields:

```python
symbol, price, currency
```

Write `quotes_from_dicts(rows)` that:

- converts dictionaries into `Quote` objects
- rejects rows with missing keys
- ignores extra keys
- returns a tuple of quotes

In [14]:
Quote = namedtuple("Quote", "symbol price currency")


def quotes_from_dicts(rows):
    quotes = []

    for index, row in enumerate(rows):
        missing = set(Quote._fields) - set(row)
        if missing:
            raise ValueError(f"row #{index} is missing required keys: {sorted(missing)}")

        quote = Quote(**{field: row[field] for field in Quote._fields})
        quotes.append(quote)

    return tuple(quotes)


api_rows = [
    {"symbol": "AAPL", "price": 190.5, "currency": "USD", "exchange": "NASDAQ"},
    {"symbol": "MSFT", "price": 420.0, "currency": "USD"}
]

quotes = quotes_from_dicts(api_rows)

assert quotes == (
    Quote("AAPL", 190.5, "USD"),
    Quote("MSFT", 420.0, "USD")
)

try:
    quotes_from_dicts([{"symbol": "TSLA", "price": 180.0}])
except ValueError as ex:
    assert "currency" in str(ex)
else:
    raise AssertionError("Expected ValueError for missing currency")

quotes

(Quote(symbol='AAPL', price=190.5, currency='USD'),
 Quote(symbol='MSFT', price=420.0, currency='USD'))

## Problem 14 — Compare Plain Tuples and Named Tuples in a Refactor

You have legacy stock tuples:

```python
(symbol, open, high, low, close)
```

Write `refactor_stocks(records)` that converts them into:

```python
StockLite(symbol, open, high, low, close)
```

Then write `price_summary(stocks)` that returns:

```python
(symbol, gain_loss, intraday_range)
```

for each stock.

Where:

```python
gain_loss = close - open
intraday_range = high - low
```

In [15]:
StockLite = namedtuple("StockLite", "symbol open high low close")
StockSummary = namedtuple("StockSummary", "symbol gain_loss intraday_range")


def refactor_stocks(records):
    return tuple(StockLite._make(record) for record in records)


def price_summary(stocks):
    return tuple(
        StockSummary(
            symbol=stock.symbol,
            gain_loss=stock.close - stock.open,
            intraday_range=stock.high - stock.low
        )
        for stock in stocks
    )


legacy = (
    ("AAPL", 190, 195, 188, 193),
    ("MSFT", 420, 425, 415, 417),
)

stocks = refactor_stocks(legacy)
summaries = price_summary(stocks)

assert stocks == (
    StockLite("AAPL", 190, 195, 188, 193),
    StockLite("MSFT", 420, 425, 415, 417)
)

assert summaries == (
    StockSummary("AAPL", 3, 7),
    StockSummary("MSFT", -3, 10)
)

summaries

(StockSummary(symbol='AAPL', gain_loss=3, intraday_range=7),
 StockSummary(symbol='MSFT', gain_loss=-3, intraday_range=10))

## Problem 15 — Design Challenge: Immutable Shopping Cart

Create two named tuples:

```python
CartItem(product_id, name, unit_price, quantity)
CartSummary(item_count, total_quantity, subtotal)
```

Write these functions:

```python
add_item(cart, item)
update_quantity(cart, product_id, quantity)
summarize_cart(cart)
```

Requirements:

- the cart itself should be represented as a tuple of `CartItem` records
- `add_item` returns a new cart
- `update_quantity` returns a new cart
- `quantity == 0` removes the item
- reject negative quantities
- `summarize_cart` returns a `CartSummary`

In [16]:
CartItem = namedtuple("CartItem", "product_id name unit_price quantity")
CartSummary = namedtuple("CartSummary", "item_count total_quantity subtotal")


def add_item(cart, item):
    if item.quantity < 0:
        raise ValueError("quantity cannot be negative")
    if item.unit_price < 0:
        raise ValueError("unit_price cannot be negative")

    return tuple(cart) + (item,)


def update_quantity(cart, product_id, quantity):
    if quantity < 0:
        raise ValueError("quantity cannot be negative")

    updated = []
    found = False

    for item in cart:
        if item.product_id == product_id:
            found = True
            if quantity > 0:
                updated.append(item._replace(quantity=quantity))
        else:
            updated.append(item)

    if not found:
        raise ValueError(f"unknown product_id: {product_id!r}")

    return tuple(updated)


def summarize_cart(cart):
    cart = tuple(cart)
    total_quantity = sum(item.quantity for item in cart)
    subtotal = sum(item.unit_price * item.quantity for item in cart)

    return CartSummary(
        item_count=len(cart),
        total_quantity=total_quantity,
        subtotal=subtotal
    )


cart = ()
cart = add_item(cart, CartItem("P100", "Keyboard", 75.00, 1))
cart = add_item(cart, CartItem("P200", "Mouse", 25.00, 2))

updated_cart = update_quantity(cart, "P200", 3)
removed_cart = update_quantity(updated_cart, "P100", 0)

assert cart != updated_cart
assert updated_cart[1].quantity == 3
assert len(removed_cart) == 1
assert removed_cart[0].product_id == "P200"
assert summarize_cart(updated_cart) == CartSummary(2, 4, 150.00)
assert summarize_cart(removed_cart) == CartSummary(1, 3, 75.00)

try:
    update_quantity(cart, "P200", -1)
except ValueError as ex:
    assert "negative" in str(ex)
else:
    raise AssertionError("Expected ValueError for negative quantity")

summarize_cart(updated_cart)

CartSummary(item_count=2, total_quantity=4, subtotal=150.0)

## Final Reflection Questions

1. Why does a named tuple behave like both a tuple and a lightweight class?
2. When is attribute access clearer than positional indexing?
3. Why does `_replace()` return a new object instead of changing the original?
4. What are the advantages and disadvantages of `rename=True`?
5. When would you choose a regular class or `dataclass` instead of a named tuple?
6. Why are named tuples useful when processing homogeneous collections of records?